<a href="https://colab.research.google.com/github/HakureiPOI/coinmetrics-fetcher/blob/main/notebooks/greeks_iv_fetcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================
# 1) 挂载 Google Drive
# =========================

# # 建议手动挂载
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# =========================
# 2) 安装依赖
# =========================
!pip -q install "git+https://github.com/HakureiPOI/coinmetrics-fetcher.git"
!pip -q install pyarrow fastparquet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 51.0 MB/s eta 0:00:00


In [3]:
# =========================
# 3) 如有 API Key，在这里设置
# =========================
import os
os.environ['COINMETRICS_API_KEY'] = '2FfPcTMqVCf0CFx6zGtJ'

In [4]:
# =========================
# 4) 导入库
# =========================
import os
import gc
import json
import time
import shutil
import traceback
from pathlib import Path

import pandas as pd

from api.fetchers.options import OptionsDataFetcher


In [5]:
# =========================
# 5) 初始化 fetcher
# =========================
options = OptionsDataFetcher()

In [6]:
# =========================
# 6) 路径设置
#    本地只做临时存储
#    Drive 只保存月度 zip
# =========================
LOCAL_ROOT = Path("/content/coinmetrics-temp")
LOCAL_MONTH_ROOT = LOCAL_ROOT / "monthly_workdir"

DRIVE_ROOT = Path("/content/drive/MyDrive/coinmetrics-data")
DRIVE_MONTHLY_ROOT = (
    DRIVE_ROOT
    / "deribit"
    / "btc"
    / "options"
    / "greeks_iv"
    / "granularity=1m"
    / "monthly_archives"
)

LOCAL_MONTH_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_MONTHLY_ROOT.mkdir(parents=True, exist_ok=True)

print("Local temp root:", LOCAL_MONTH_ROOT)
print("Drive monthly archive root:", DRIVE_MONTHLY_ROOT)

Local temp root: /content/coinmetrics-temp/monthly_workdir
Drive monthly archive root: /content/drive/MyDrive/coinmetrics-data/deribit/btc/options/greeks_iv/granularity=1m/monthly_archives


In [7]:
# =========================
# 7) 只保留核心字段
#    按你项目的真实返回列结构设置
# =========================
KEEP_COLUMNS = [
    "market",
    "time",
    "vega",
    "theta",
    "rho",
    "delta",
    "gamma",
    "iv_bid",
    "iv_ask",
    "iv_mark",
    "strike",
    "expiration",
    "option_contract_type",
    "listing",
]

In [8]:
# =========================
# 8) 压缩数据类型
# =========================
def optimize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # time 保持 UTC-aware（原始通常就是带 UTC 的）
    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True)

    # expiration / listing 保持 naive datetime
    for col in ["expiration", "listing"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # 浮点列压成 float32
    float_cols = [
        "vega", "theta", "rho", "delta", "gamma",
        "iv_bid", "iv_ask", "iv_mark"
    ]
    for col in float_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

    # strike 压缩
    if "strike" in df.columns:
        strike_num = pd.to_numeric(df["strike"], errors="coerce")
        if strike_num.isna().any():
            df["strike"] = strike_num.astype("float32")
        else:
            df["strike"] = strike_num.astype("int32")

    # 类别列压缩
    for col in ["market", "option_contract_type"]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    # 排序
    sort_cols = [c for c in ["time", "market"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)

    return df

In [9]:
# =========================
# 9) 抓取单日数据
# =========================
def fetch_one_day(
    start_time: pd.Timestamp,
    end_time: pd.Timestamp,
    batch_size: int = 100,
    max_workers: int = 4,
) -> pd.DataFrame:
    # 关键：用 ISO 8601，但不带时区
    start_str = start_time.strftime("%Y-%m-%dT%H:%M:%S")
    end_str = end_time.strftime("%Y-%m-%dT%H:%M:%S")

    df = options.get_options_greeks_iv(
        exchange="deribit",
        base="btc",
        start_time=start_str,
        end_time=end_str,
        granularity="1m",
        batch_size=batch_size,
        max_workers=max_workers,
    )

    if df is None or len(df) == 0:
        return pd.DataFrame(columns=KEEP_COLUMNS)

    existing_cols = [c for c in KEEP_COLUMNS if c in df.columns]
    df = df[existing_cols].copy()
    df = optimize_dataframe(df)
    return df

In [10]:
# =========================
# 10) 处理单个月
#     按天抓 -> 本地 daily parquet
#     合并成单个月度 parquet
#     压成 zip -> 上传 Drive
#     删除本地临时文件
# =========================
def process_one_month(
    year: int,
    month: int,
    batch_size: int = 100,
    max_workers: int = 4,
    sleep_sec: int = 2,
    overwrite_drive_zip: bool = False,
):
    year_str = f"{year:04d}"
    month_str = f"{month:02d}"

    drive_zip_path = DRIVE_MONTHLY_ROOT / f"deribit-btc-options-greeks-iv-1m-{year_str}{month_str}.zip"
    if drive_zip_path.exists() and not overwrite_drive_zip:
        print(f"[SKIP-MONTH] {year_str}-{month_str} already exists on Drive")
        return {
            "year": year_str,
            "month": month_str,
            "status": "skipped",
            "drive_zip": str(drive_zip_path),
            "rows": None,
        }

    month_workdir = LOCAL_MONTH_ROOT / f"{year_str}{month_str}"
    daily_dir = month_workdir / "daily_parts"
    daily_dir.mkdir(parents=True, exist_ok=True)

    # naive datetime，适配你项目内部 listing/expiration 的比较逻辑
    month_start = pd.Timestamp(f"{year_str}-{month_str}-01")
    next_month_start = month_start + pd.offsets.MonthBegin(1)
    day_starts = pd.date_range(month_start, next_month_start, freq="D")

    failed_days = []
    total_rows = 0
    daily_summary = []

    print(f"\n========== Processing {year_str}-{month_str} ==========")

    # 逐日抓取并写本地 parquet
    for i in range(len(day_starts) - 1):
        start_time = day_starts[i]
        end_time = day_starts[i + 1]
        date_str = start_time.strftime("%Y-%m-%d")
        local_day_file = daily_dir / f"deribit-btc-options-greeks-iv-1m-{date_str}.parquet"

        try:
            print(f"[FETCH] {date_str}")
            df_day = fetch_one_day(
                start_time=start_time,
                end_time=end_time,
                batch_size=batch_size,
                max_workers=max_workers,
            )

            rows = len(df_day)
            total_rows += rows

            df_day.to_parquet(
                local_day_file,
                engine="pyarrow",
                compression="zstd",
                index=False
            )

            daily_summary.append({
                "date": date_str,
                "rows": rows,
                "file": local_day_file.name,
            })

            print(f"[DONE] {date_str}: rows={rows:,}")

            del df_day
            gc.collect()
            time.sleep(sleep_sec)

        except Exception as e:
            failed_days.append({
                "date": date_str,
                "error": repr(e),
                "traceback": traceback.format_exc(),
            })
            print(f"[FAILED] {date_str}: {e}")

    # 有失败天就不上传整月 zip
    if failed_days:
        fail_log = month_workdir / f"failed_days_{year_str}{month_str}.json"
        with open(fail_log, "w", encoding="utf-8") as f:
            json.dump(failed_days, f, ensure_ascii=False, indent=2)

        summary_path = month_workdir / f"daily_summary_{year_str}{month_str}.json"
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(daily_summary, f, ensure_ascii=False, indent=2)

        print(f"[MONTH-FAILED] {year_str}-{month_str} has failed days, skip upload")
        print(f"Failed log saved to: {fail_log}")

        return {
            "year": year_str,
            "month": month_str,
            "status": "partial_failed",
            "drive_zip": None,
            "rows": total_rows,
            "failed_days": len(failed_days),
            "failed_log": str(fail_log),
        }

    # 没有任何日文件
    part_files = sorted(daily_dir.glob("*.parquet"))
    if len(part_files) == 0:
        print(f"[EMPTY-MONTH] {year_str}-{month_str}")
        shutil.rmtree(month_workdir, ignore_errors=True)
        return {
            "year": year_str,
            "month": month_str,
            "status": "empty",
            "drive_zip": None,
            "rows": 0,
        }

    # 保存月度元信息（放进 zip 一起归档）
    meta = {
        "year": year_str,
        "month": month_str,
        "rows": int(total_rows),
        "num_daily_files": len(part_files),
        "columns": KEEP_COLUMNS,
        "files": [p.name for p in part_files],
    }
    meta_path = month_workdir / f"deribit-btc-options-greeks-iv-1m-{year_str}{month_str}-meta.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    summary_path = month_workdir / f"daily_summary_{year_str}{month_str}.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(daily_summary, f, ensure_ascii=False, indent=2)

    # 直接把整个 month_workdir 打成 zip
    # zip 内将包含:
    # - daily_parts/*.parquet
    # - meta.json
    # - daily_summary.json
    zip_base = LOCAL_MONTH_ROOT / f"deribit-btc-options-greeks-iv-1m-{year_str}{month_str}"
    local_zip_path = shutil.make_archive(
        base_name=str(zip_base),
        format="zip",
        root_dir=str(month_workdir)
    )
    local_zip_path = Path(local_zip_path)
    print(f"[ZIP-DONE] {local_zip_path}")

    shutil.copy2(local_zip_path, drive_zip_path)
    print(f"[UPLOAD-DONE] {drive_zip_path}")

    # 清理本地临时文件
    shutil.rmtree(month_workdir, ignore_errors=True)
    if local_zip_path.exists():
        local_zip_path.unlink()

    gc.collect()
    print(f"[CLEANUP-DONE] {year_str}-{month_str}")

    return {
        "year": year_str,
        "month": month_str,
        "status": "ok",
        "drive_zip": str(drive_zip_path),
        "rows": total_rows,
        "failed_days": 0,
        "num_daily_files": len(part_files),
    }

In [ ]:
# =========================
# 跑 2025 全年
# =========================
all_month_results = []

for month in range(1, 13):
    result = process_one_month(
        year=2025,
        month=month,
        batch_size=100,
        max_workers=4,
        sleep_sec=2,
        overwrite_drive_zip=False,   # 如果 Drive 上已有同名 zip，就跳过
    )
    all_month_results.append(result)

    # 每处理完一个月就把阶段性汇总写到 Drive
    summary_df = pd.DataFrame(all_month_results)
    summary_csv = DRIVE_MONTHLY_ROOT / "monthly_download_summary_2025.csv"
    summary_df.to_csv(summary_csv, index=False)

    print(f"\n[PROGRESS] finished month={month:02d}")
    print(summary_df.tail(3))

print("\n==========================")
print("All months finished.")
print("==========================")

summary_df = pd.DataFrame(all_month_results)
summary_csv = DRIVE_MONTHLY_ROOT / "monthly_download_summary_2025.csv"
summary_df.to_csv(summary_csv, index=False)

print(summary_df)
print("Summary saved to:", summary_csv)


========== Processing 2025-01 ==========
[FETCH] 2025-01-01
06:35:47 INFO    [Greeks/IV] DERIBIT BTC | 840 个期权
06:36:07 INFO    [Greeks]  2/9 | 累计  94132 条
06:36:14 INFO    [Greeks]  4/9 | 累计 238132 条
06:36:16 INFO    [Greeks]  3/9 | 累计 382132 条
06:36:16 INFO    [Greeks]  1/9 | 累计 526132 条
06:36:31 INFO    [Greeks]  5/9 | 累计 670132 条
06:36:36 INFO    [Greeks]  6/9 | 累计 808464 条
06:36:39 INFO    [Greeks]  9/9 | 累计 846572 条
06:36:40 INFO    [Greeks]  7/9 | 累计 990572 条
06:36:42 INFO    [Greeks]  8/9 | 累计 1134572 条
06:36:42 INFO    [Greeks] 完成：9/9 批次，共 1134572 条
06:36:59 INFO    [IV]  2/9 | 累计  94132 条
06:37:01 INFO    [IV]  3/9 | 累计 238132 条
06:37:02 INFO    [IV]  1/9 | 累计 382132 条
06:37:05 INFO    [IV]  4/9 | 累计 526132 条
06:37:17 INFO    [IV]  5/9 | 累计 670132 条
06:37:19 INFO    [IV]  7/9 | 累计 814132 条
06:37:21 INFO    [IV]  6/9 | 累计 952464 条
06:37:22 INFO    [IV]  9/9 | 累计 990572 条
06:37:25 INFO    [IV]  8/9 | 累计 1134572 条
06:37:25 INFO    [IV] 完成：9/9 批次，共 1134572 条
06:37:26 INFO    [Gr